# Phase 3: Query Set Development — Types 1, 2, 5, 6

> **Scope of this notebook:** Auto-generate queries for **Types 1, 2, 5, and 6** only.
> These types can be derived directly from the dataset with no manual input required.
>
> **Query Types 3, 4, and 7 are out of scope here.**
> They require domain expertise and must be created **entirely manually**:
> - **Type 3 (Semantic/Descriptive):** an analyst must write natural language descriptions — no entity name or identifier
> - **Type 4 (Relational/Graph):** requires understanding entity relationships and formulating link-based questions
> - **Type 7 (RAG Summarisation):** both the query and the gold-standard answer paragraph must be written by a domain expert

---

**Output:** `data/evaluation/potential_queries_type_1_2_5_6.xlsx` — 200 queries (50 per type)

Each row has:
- `query_id` — unique identifier
- `query_type` — 1, 2, 5, or 6
- `query_texts` — JSON array: `["primary text", "alt variant 1", "alt variant 2"]`
- `filter_criteria` — Type 6 only: `programId=X | schema=Y | country=Z` (used by ground truth script)
- `notes` — human-readable context

In [ ]:
# ── Shared Setup ──────────────────────────────────────────────────────────────
import json, os, re, sys, random
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 60)
random.seed(42)

# ── Locate project root ───────────────────────────────────────────────────────
def _find_root():
    marker = 'data/raw_data'
    # 1. Env vars set by IDE
    for key in ('IR_PROJECT_ROOT', 'VSCODE_WORKSPACE_FOLDER', 'CURSOR_WORKSPACE_FOLDER'):
        v = os.environ.get(key)
        if v and (Path(v) / marker).exists():
            return Path(v)
    # 2. Walk up from current working directory
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / marker).exists():
            return p
    # 3. Notebook path injected by VS Code / Cursor
    nb = globals().get('__vsc_ipynb_file__') or os.environ.get('JPY_SESSION_NAME', '')
    if nb:
        for p in Path(nb).parents:
            if (p / marker).exists():
                return p
    # 4. Hardcoded fallback — update this if the project moves
    fallback = Path("/Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_temr_2/IR_project")
    if (fallback / marker).exists():
        return fallback
    raise FileNotFoundError("Cannot find project root. Set IR_PROJECT_ROOT env var.")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FULL     = ROOT / 'data' / 'processed' / 'full'        / 'documents.jsonl'
SUBSET   = ROOT / 'data' / 'processed' / 'subset_100k' / 'documents.jsonl'
RAW_FULL = ROOT / 'data' / 'raw_data'  / 'targets.nested.json'
EVAL_DIR = ROOT / 'data' / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DOCS_PATH = FULL if FULL.exists() else SUBSET
label     = 'FULL' if FULL.exists() else '100K SUBSET'
print(f'Root    : {ROOT}')
print(f'Dataset : {label}  →  {DOCS_PATH}')
if not FULL.exists():
    print('NOTE: re-run after the full pipeline finishes for production queries.')


---
# PART A — Auto-Generated Queries
### Types 1 · 2 · 5 · 6 — 50 queries each, with two alternatives per query

All queries are derived directly from real entities in `documents.jsonl`.  
No manual input is required; the human role is only to **review and trim** the Excel output.

In [2]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def typo_swap(s: str) -> str:
    """Swap two adjacent characters near the middle of s."""
    if len(s) < 4:
        return s + s[-1]  # double last char
    mid = len(s) // 2
    lst = list(s)
    lst[mid], lst[mid + 1] = lst[mid + 1], lst[mid]
    return ''.join(lst)

def typo_drop(s: str) -> str:
    """Drop one character near the end."""
    if len(s) < 3:
        return s
    idx = max(len(s) - 2, 1)
    return s[:idx] + s[idx + 1:]

def typo_double(s: str) -> str:
    """Double one character near the middle."""
    if len(s) < 2:
        return s
    mid = len(s) // 2
    return s[:mid] + s[mid] + s[mid:]

def id_swap_digits(s: str) -> str:
    """For identifier strings: swap the last two digit characters."""
    digits = [(i, c) for i, c in enumerate(s) if c.isdigit()]
    if len(digits) < 2:
        return s + '0'
    i1, i2 = digits[-2][0], digits[-1][0]
    lst = list(s)
    lst[i1], lst[i2] = lst[i2], lst[i1]
    return ''.join(lst)

def id_space_after_prefix(s: str) -> str:
    """Insert a space after a letter prefix (e.g. IMO9... → IMO 9...)."""
    m = re.match(r'^([A-Za-z]+)(\d.*)', s)
    if m:
        return m.group(1) + ' ' + m.group(2)
    return s.lower()

def name_order_flip(s: str) -> str:
    """
    Generate a word-order permutation of a name string — the most realistic
    real-world variation for Type 2 queries.

    Rules:
      - If already in 'Last, First' comma format → flip to 'First Last'
          'FASONU, AYODEJI'   → 'AYODEJI FASONU'
          'Melia, Patricia'   → 'Patricia Melia'
      - Otherwise move the last word to the front (surname-first convention):
          'Kan Chen'                        → 'Chen Kan'
          'Arnita Cowan Leff'               → 'Leff Arnita Cowan'
          'Yatai Smart Industrial New City' → 'City Yatai Smart Industrial New'
      - Single-word strings are returned unchanged.
    """
    if ',' in s:
        parts = [p.strip() for p in s.split(',', 1)]
        return f'{parts[1]} {parts[0]}'.strip()
    words = s.split()
    if len(words) >= 2:
        return f'{words[-1]} {" ".join(words[:-1])}'
    return s

print('Helpers loaded.')

Helpers loaded.


---
## Type 1 — Exact Identifier Queries

Pick real entities with known IMO numbers, MMSI, call signs, or registration numbers.  
Primary query = the raw identifier string.  
Alt 1 = last two digits swapped.  
Alt 2 = space inserted after letter prefix (or lowercase).

In [3]:
TARGET_N = 50
rows_t1  = []

with open(DOCS_PATH) as f:
    for line in f:
        doc = json.loads(line)
        ids = doc.get('identifiers', {})
        if not ids:
            continue
        for id_type, values in ids.items():
            for val in values:
                val = str(val).strip()
                if not val:
                    continue
                rows_t1.append({
                    'query_id':          f'Q1_{len(rows_t1)+1:03d}',
                    'query_type':        1,
                    'query_text':        val,
                    'alt_query_1':       id_swap_digits(val),
                    'alt_query_2':       id_space_after_prefix(val),
                    'expected_doc_ids':  doc['doc_id'],
                    'notes':             f'{id_type} | {doc["caption"]} [{doc["schema"]}]',
                })
                if len(rows_t1) >= TARGET_N:
                    break
            if len(rows_t1) >= TARGET_N:
                break
        if len(rows_t1) >= TARGET_N:
            break

df_t1 = pd.DataFrame(rows_t1)
print(f'Type 1: {len(df_t1)} queries generated')
print(df_t1[['query_id', 'query_text', 'alt_query_1', 'alt_query_2', 'notes']].head(10).to_string(index=False))

Type 1: 50 queries generated
query_id    query_text   alt_query_1   alt_query_2                                                                                                                         notes
  Q1_001     103919088     103919088     103919088                                            registrationNumber | Myanmar Yatai International Holding Group Co., LTD. [Company]
  Q1_002  PW2XZT68KVW9  PW2XZT69KVW8 PW 2XZT68KVW9                                                uniqueEntityId | Myanmar Yatai International Holding Group Co., LTD. [Company]
  Q1_003  PW3LMJ5YB3M3  PW3LMJ5YB3M3 PW 3LMJ5YB3M3                                                uniqueEntityId | Myanmar Yatai International Holding Group Co., LTD. [Company]
  Q1_004  PW2USM6LNCJ9  PW2USM9LNCJ6 PW 2USM6LNCJ9                                                uniqueEntityId | Myanmar Yatai International Holding Group Co., LTD. [Company]
  Q1_005  GQBPAV1TFF41  GQBPAV1TFF14 GQBPAV 1TFF41                                    

---
## Type 2 — Name / Alias Queries

Find entities with multiple name variants in the raw data.  
Primary query = a *non-caption* alias (so the test is non-trivial).  
Alt 1 = **word-order permutation** (surname-first / last-word-first) — tests systems that depend on token order.  
Alt 2 = **character-swap typo** — tests fuzzy name matching.

In [4]:
# Stream raw data to collect entities with real aliases
# We need the raw file because documents.jsonl only has text_blob, not the original alias list
from src.preprocessing.parser import stream_records

alias_pool = []   # (doc_id, caption, alias)
SCAN_LIMIT = 200_000  # scan first 200K raw records — enough to find 50 good candidates

alias_fields = ['alias', 'previousName', 'weakAlias', 'name']

for rec in stream_records(RAW_FULL, max_records=SCAN_LIMIT, show_progress=True):
    props = rec.get('properties', {})
    caption = rec.get('caption', '')
    # Collect aliases that differ meaningfully from the caption
    for field in alias_fields:
        for alias_raw in props.get(field, []):
            alias = str(alias_raw).strip() if isinstance(alias_raw, str) else ''
            if not alias:
                continue
            # Skip if alias == caption (that is not a useful test)
            if alias.lower() == caption.lower():
                continue
            # Skip very short aliases
            if len(alias) < 5:
                continue
            alias_pool.append({
                'doc_id':  rec['id'],
                'caption': caption,
                'schema':  rec['schema'],
                'field':   field,
                'alias':   alias,
            })
    if len(alias_pool) >= TARGET_N * 10:
        break

# Sample 50 unique doc_ids (prefer entities with the most aliases)
df_alias_pool = pd.DataFrame(alias_pool)
print(f'Total alias entries collected: {len(df_alias_pool)}')

seen_ids = set()
rows_t2  = []
for _, row in df_alias_pool.iterrows():
    doc_id = row['doc_id']
    if doc_id in seen_ids:
        continue
    seen_ids.add(doc_id)
    alias = row['alias']
    rows_t2.append({
            'query_id':         f'Q2_{len(rows_t2)+1:03d}',
            'query_type':       2,
            'query_text':       alias,
            'alt_query_1':      name_order_flip(alias),   # word-order permutation
            'alt_query_2':      typo_swap(alias),          # character-swap typo
            'expected_doc_ids': doc_id,
            'notes':            f'{row["field"]} | caption: "{row["caption"]}" [{row["schema"]}]',
        })
    if len(rows_t2) >= TARGET_N:
        break

df_t2 = pd.DataFrame(rows_t2)
print(f'\nType 2: {len(df_t2)} queries generated')
print(df_t2[['query_id', 'query_text', 'alt_query_1', 'alt_query_2', 'notes']].head(10).to_string(index=False))

Streaming records: 238 lines [00:00, 23590.23 lines/s]

Total alias entries collected: 502

Type 2: 50 queries generated
query_id                                                  query_text                                                 alt_query_1                                                alt_query_2                                                                                                                       notes
  Q2_001                             Yatai Smart Industrial New City                             Yatai Smart Indsutrial New City                             Yatai Smart Industrial New Ciy                                            alias | caption: "Myanmar Yatai International Holding Group Co., LTD." [Company]
  Q2_002 Tovarystvo z obmezhenoiu vidpovidalnistiu "Zelinskyi Hrupp" Tovarystvo z obmezhenoiu vidpvoidalnistiu "Zelinskyi Hrupp" Tovarystvo z obmezhenoiu vidpovidalnistiu "Zelinskyi Hrup"                                     alias | caption: "Товариство з обмеженою відповідальністю "Зелінський Групп"" [Company]

---
## Type 5 — Cross-Dataset Deduplication Queries

Find entities that have **multiple distinct doc_ids** for the same name — i.e. the same real-world entity
has not been fully merged across datasets and exists as separate records.  
A correct system must retrieve **all** of them.  
Primary query = the shared caption.  
Alt 1 = lowercase. Alt 2 = drop the last word.  
`n_relevant` = number of distinct doc_ids sharing that caption (always ≥ 2 here).

In [5]:
# Pass 1: group all doc_ids by caption — find captions with MULTIPLE doc_ids
# (these are true un-merged duplicates that the system must retrieve together)
from collections import defaultdict

caption_to_docs = defaultdict(list)  # caption → [(doc_id, schema, datasets_str)]

with open(DOCS_PATH) as f:
    for line in f:
        doc = json.loads(line)
        cap = doc['caption'].strip()
        if not cap:
            continue
        caption_to_docs[cap].append({
            'doc_id':   doc['doc_id'],
            'schema':   doc['schema'],
            'datasets': '; '.join(doc['metadata'].get('datasets', [])[:4]),
        })

# Generic captions to skip — not meaningful as queries
GENERIC_CAPTIONS = {
    'person', 'company', 'organization', 'vessel', 'security', 'entity',
    'unknown', 'name', 'individual', 'cryptocurrency wallet', 'wallet',
    'address', 'account',
}

def is_meaningful_caption(cap: str) -> bool:
    if len(cap) < 6:
        return False
    if cap.lower() in GENERIC_CAPTIONS:
        return False
    if not any(c.isalpha() for c in cap):
        return False
    # Skip hex hashes
    if len(cap) > 30 and all(c in '0123456789abcdefABCDEF' for c in cap.replace(' ', '')):
        return False
    return True

# Keep only captions with >= 2 distinct doc_ids and a meaningful name
true_dupes = {
    cap: docs
    for cap, docs in caption_to_docs.items()
    if len({d['doc_id'] for d in docs}) >= 2
    and is_meaningful_caption(cap)
}
print(f'Captions with >=2 doc_ids (true duplicates, meaningful name): {len(true_dupes)}')

# Sort by number of distinct doc_ids descending
sorted_dupes = sorted(
    true_dupes.items(),
    key=lambda x: len({d['doc_id'] for d in x[1]}),
    reverse=True
)

rows_t5 = []
for caption, docs in sorted_dupes[:TARGET_N]:
    distinct_ids = list({d['doc_id'] for d in docs})
    n_rel        = len(distinct_ids)
    words        = caption.split()
    alt2         = ' '.join(words[:-1]) if len(words) > 1 else caption[:-1]
    schemas      = ', '.join({d['schema'] for d in docs})
    rows_t5.append({
        'query_id':         f'Q5_{len(rows_t5)+1:03d}',
        'query_type':       5,
        'query_text':       caption,
        'alt_query_1':      caption.lower(),
        'alt_query_2':      alt2,
        'expected_doc_ids': '; '.join(distinct_ids),
        'n_relevant':       n_rel,
        'notes':            f'{n_rel} doc_ids for same caption [{schemas}]',
    })

df_t5 = pd.DataFrame(rows_t5)
print(f'\nType 5: {len(df_t5)} queries generated')
print(df_t5[['query_id', 'query_text', 'n_relevant', 'notes']].head(10).to_string(index=False))

Captions with >=2 doc_ids (true duplicates, meaningful name): 32566

Type 5: 50 queries generated
query_id                                                                                                   query_text  n_relevant                                     notes
  Q5_001                                                                                             Morten Austestad         102     102 doc_ids for same caption [Person]
  Q5_002                                                                                               Per Atle Tufte          74      74 doc_ids for same caption [Person]
  Q5_003                                                                                        Jørn Steinar Seljelid          69      69 doc_ids for same caption [Person]
  Q5_004                                  Биржевые облигации процентные неконвертируемые бездокументарные серии БО-01          63    63 doc_ids for same caption [Security]
  Q5_005 Биржевые облигации процентные нек

---
## Type 6 — Jurisdiction / Filter Queries

Scoped queries: sanction programme + entity type + country.  
Unlike Types 1/2, these match **many** documents — sorted by `n_relevant` descending.  
`expected_doc_ids` stores a representative sample (up to 20); `n_relevant` gives the true total.  
The `filter_criteria` column defines relevance for the evaluation framework.  
Alt 1 = synonym swap (vessel↔ship). Alt 2 = ISO country code instead of full name.

In [6]:
# Build programId × schema × country → ALL matching doc_ids
combo_docs = defaultdict(list)   # (prog, schema, country) → [doc_ids]

COUNTRY_NAME = {
    'ru': 'Russia',      'kp': 'North Korea', 'ir': 'Iran',       'sy': 'Syria',
    'cn': 'China',       'by': 'Belarus',     'mm': 'Myanmar',    've': 'Venezuela',
    'ua': 'Ukraine',     'ly': 'Libya',       'so': 'Somalia',    'sd': 'Sudan',
    'iq': 'Iraq',        'af': 'Afghanistan', 'cu': 'Cuba',       'zw': 'Zimbabwe',
    'us': 'US',          'br': 'Brazil',      'pk': 'Pakistan',   'ng': 'Nigeria',
}
SCHEMA_WORD = {'Vessel': 'vessel', 'Person': 'person', 'Company': 'company',
               'LegalEntity': 'organization', 'Organization': 'organization'}
SCHEMA_ALT  = {'Vessel': 'ship',   'Person': 'individual', 'Company': 'entity',
               'LegalEntity': 'entity', 'Organization': 'entity'}

with open(DOCS_PATH) as f:
    for line in f:
        doc  = json.loads(line)
        meta = doc['metadata']
        progs = meta.get('programId', [])
        ctrs  = meta.get('country',   [])
        sch   = doc['schema']
        if not progs or not ctrs:
            continue
        for prog in set(progs):       # deduplicate within doc
            for ctr in set(ctrs):
                combo_docs[(prog, sch, ctr)].append(doc['doc_id'])

# Sort by n_relevant descending, keep top TARGET_N with known countries
rows_t6     = []
seen_combos = set()

for (prog, sch, ctr), doc_ids in sorted(combo_docs.items(), key=lambda x: -len(x[1])):
    if ctr not in COUNTRY_NAME:
        continue
    if len(doc_ids) < 5:
        continue
    key = (prog, sch, ctr)
    if key in seen_combos:
        continue
    seen_combos.add(key)

    country    = COUNTRY_NAME[ctr]
    s_word     = SCHEMA_WORD.get(sch, sch.lower())
    s_alt      = SCHEMA_ALT.get(sch, sch.lower())
    prog_clean = prog.replace('-', ' ')
    n_rel      = len(doc_ids)

    rows_t6.append({
        'query_id':         f'Q6_{len(rows_t6)+1:03d}',
        'query_type':       6,
        'query_text':       f'{prog_clean} {s_word} {country}',
        'alt_query_1':      f'{prog_clean} {s_alt} {country}',
        'alt_query_2':      f'{prog_clean} {s_word} {ctr.upper()}',
        'expected_doc_ids': '; '.join(doc_ids[:20]),   # sample — true set defined by filter_criteria
        'n_relevant':       n_rel,
        'filter_criteria':  f'programId={prog} | schema={sch} | country={ctr}',
        'notes':            f'{n_rel} matching docs',
    })
    if len(rows_t6) >= TARGET_N:
        break

df_t6 = pd.DataFrame(rows_t6)
print(f'Type 6: {len(df_t6)} queries generated')
print(df_t6[['query_id', 'query_text', 'n_relevant', 'filter_criteria']].head(15).to_string(index=False))

Type 6: 50 queries generated
query_id                 query_text  n_relevant                                     filter_criteria
  Q6_001       US HHS OIG person US       78312   programId=US-HHS-OIG | schema=Person | country=us
  Q6_002     BR CEIS company Brazil        9045     programId=BR-CEIS | schema=Company | country=br
  Q6_003      BR CEIS person Brazil        7231      programId=BR-CEIS | schema=Person | country=br
  Q6_004    UA SA1644 person Russia        6165    programId=UA-SA1644 | schema=Person | country=ru
  Q6_005 PK ATA1997 person Pakistan        4206   programId=PK-ATA1997 | schema=Person | country=pk
  Q6_006      US HHS OIG company US        3280  programId=US-HHS-OIG | schema=Company | country=us
  Q6_007   US RUSHAR company Russia        3050   programId=US-RUSHAR | schema=Company | country=ru
  Q6_008   UA SA1644 company Russia        2955   programId=UA-SA1644 | schema=Company | country=ru
  Q6_009  INTERPOL RN person Russia        2752  programId=INTERPOL-RN 

---
## Part A Summary & Export

In [7]:
# Add n_relevant to Types 1 and 2 (always 1 — single-target queries)
df_t1['n_relevant'] = 1
df_t2['n_relevant'] = 1
# df_t5 and df_t6 already have n_relevant

# Ensure filter_criteria column exists in all frames
for df in [df_t1, df_t2, df_t5]:
    if 'filter_criteria' not in df.columns:
        df['filter_criteria'] = ''

# Combine and sort by n_relevant descending within each type
df_part_a = pd.concat([df_t1, df_t2, df_t5, df_t6], ignore_index=True)
df_part_a = df_part_a.sort_values(['query_type', 'n_relevant'], ascending=[True, False])
df_part_a = df_part_a.reset_index(drop=True)

print('=== Part A Summary ===')
summary = df_part_a.groupby('query_type')['n_relevant'].agg(['count', 'min', 'max', 'median'])
summary.index = ['Type 1 (Identifier)', 'Type 2 (Alias)', 'Type 5 (Cross-Dataset)', 'Type 6 (Jurisdiction)']
summary.columns = ['n_queries', 'min_relevant', 'max_relevant', 'median_relevant']
print(summary.to_string())
print(f'\nTotal: {len(df_part_a)} queries')

=== Part A Summary ===
                        n_queries  min_relevant  max_relevant  median_relevant
Type 1 (Identifier)            50             1             1              1.0
Type 2 (Alias)                 50             1             1              1.0
Type 5 (Cross-Dataset)         50            15           102             24.5
Type 6 (Jurisdiction)          50           269         78312            523.5

Total: 200 queries


In [9]:
import json as _json

# Build clean queries dataframe with array-format query_texts
def make_query_texts(row):
    """Combine primary + alternatives into a JSON array, skipping empty variants."""
    texts = [row['query_text']]
    for alt in ['alt_query_1', 'alt_query_2']:
        val = str(row.get(alt, '')).strip()
        if val and val != row['query_text']:
            texts.append(val)
    return _json.dumps(texts, ensure_ascii=False)

df_export = pd.DataFrame({
    'query_id':        df_part_a['query_id'],
    'query_type':      df_part_a['query_type'],
    'query_texts':     df_part_a.apply(make_query_texts, axis=1),
    'filter_criteria': df_part_a.get('filter_criteria', pd.Series([''] * len(df_part_a))),
    'notes':           df_part_a.get('notes',            pd.Series([''] * len(df_part_a))),
})

# Export to Excel
out_a = EVAL_DIR / 'potential_queries_type_1_2_5_6.xlsx'

from openpyxl.styles import PatternFill, Font
header_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')

with pd.ExcelWriter(out_a, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Queries', index=False)
    ws = writer.sheets['Queries']
    widths = {'A': 12, 'B': 12, 'C': 90, 'D': 50, 'E': 60}
    for col, w in widths.items():
        ws.column_dimensions[col].width = w
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill

print(f'Exported {len(df_export)} queries → {out_a}')
print()
print('Column guide:')
print('  query_id         = unique identifier')
print('  query_type       = 1, 2, 5, or 6')
print('  query_texts      = JSON array: ["primary text", "alt variant 1", "alt variant 2"]')
print('  filter_criteria  = Type 6 only: programId=X | schema=Y | country=Z (used by qrels script)')
print('  notes            = human-readable context / source field info')
print()
print('Human review instructions:')
print('  1. Delete rows where query_texts looks wrong or trivial')
print('  2. Edit the JSON array in query_texts to fix or add variants')
print('  3. Save the file when done — no rename needed')

Exported 200 queries → /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_tem_2/IR_project/data/evaluation/queries_part_a.xlsx

Column guide:
  n_relevant        = number of expected relevant docs for this query
  expected_doc_ids  = sample of relevant doc_ids (Types 1/2: complete; Type 6: top 20)
  filter_criteria   = machine-readable relevance definition (Type 6 only)

Human review instructions:
  1. Sort by n_relevant to see the richest queries first
  2. Delete rows where query_text looks wrong or trivial
  3. Fix alt_query variants identical to query_text
  4. Save as queries_part_a_reviewed.xlsx


Type 3 candidates: 100

                                                                               caption       schema        country                                keywords  n_tokens
                                                                Mr Murtaza Ali Lakhani       Person ca, mc, pk, gb crude, launder, oil, prohibit, sanction      1293
                                                  Limited Liability Company Molot Arms      Company             ru                  arms, financ, sanction      1231
                                                        Nikolai Aleksandrovich Tupikin       Person             ru               financ, launder, sanction      1090
                                                         Anatoliy Anatolievich Bulavko       Person             by       financ, launder, sanction, weapon      1045
                Joint Stock Company State Research Institute of Instrument Engineering      Company             ru                financ, sanction, wea

[Person] Mr Murtaza Ali Lakhani (ca, mc, pk, gb) — 1293 tokens
  Preview: mr murtaza ali lakhani lakhani murtaza ali лахані муртаза алі murtaza ali lakhani murtaza lakhani lakhani murtaza ali businessperson homme d affaires sanction corp disqual ua sa1644 eu ukr seco ukrain

[Company] Limited Liability Company Molot Arms (ru) — 1231 tokens
  Preview: molot arms korlatolt felelossegu tarsasag общество ограниченнои ответственностью молот армз товариство обмеженою відповідальністю молот армз общество ограниченнои ответственностью молот армз общество 

[Person] Nikolai Aleksandrovich Tupikin (ru) — 1090 tokens
  Preview: tupikin nikolai aleksandrovich тупикин николаи александрович nikolai tupikin nikolai aleksandrovich tupikin тупікін микола олександрович tupikin nikolai tupikin nikolai aleksandrovich nikolay aleksand

[Person] Anatoliy Anatolievich Bulavko (by) — 1045 tokens
  Preview: anatoliy anatolievich bulavko anatoliy bulavko анатоль анатольевiч булаука булавко анатоліи анатоліиович

Streaming records: 1004 lines [00:00, 15457.03 lines/s]

Type 4 candidates: 200

                                                                                               caption  schema  ownership_count  directorship_count                                                     datasets
                                                                  RT-Capital Limited Liability Company Company               30                   2 us_trade_csl, nz_russia_sanctions, us_sam_exclusions, gem_en
Регіональне об'єднання роботодавців "Спілка промисловців і підприємців Луганської Народної Республіки" Company               25                   1                              ua_nsdc_sanctions, ext_ru_egrul
                                            ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ "НОРДЭНЕРГОГРУПП" Company               13                   3                               ext_ru_egrul, ann_graph_topics
                                                                                Russian Universal Bank Company                8             

Type 7 candidates: 200

                                                                                                                           caption       schema  n_tokens  n_datasets  n_ids  score
                                                                                          International Investment Platform ocp as      Company      2608           2      0   2628
МИНИСТЕРСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ ПО ДЕЛАМ ГРАЖДАНСКОЙ ОБОРОНЫ, ЧРЕЗВЫЧАЙНЫМ СИТУАЦИЯМ И ЛИКВИДАЦИИ ПОСЛЕДСТВИЙ СТИХИЙНЫХ БЕДСТВИЙ      Company      1391           2      7   1446
                                                                                                       AL-IHSAN CHARITABLE SOCIETY Organization      1197           6     16   1337
                                                                                                     Anatoliy Anatolievich Bulavko       Person      1045          12      0   1165
                                                                            

[Company] International Investment Platform ocp as  (score=2628, datasets=2)
  international investment platform international investment platform ocp as reg warn national bank slovakia nbs bankova rada narodnej banky slovenska ako druhostupnovy organ prislusny podla od zakona d

[Company] МИНИСТЕРСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ ПО ДЕЛАМ ГРАЖДАНСКОЙ ОБОРОНЫ, ЧРЕЗВЫЧАЙНЫМ СИТУАЦИЯМ И ЛИКВИДАЦИИ ПОСЛЕДСТВИЙ СТИХИЙНЫХ БЕДСТВИЙ  (score=1446, datasets=2)
  министерство россиискои федерации по делам гражданскои обороны чрезвычаиным ситуациям ликвидациям последствии стихииных бедствии министерство россиискои федерации по делам гражданскои обороны чрезвыча

[Organization] AL-IHSAN CHARITABLE SOCIETY  (score=1337, datasets=6)
  the benevolent charitable organization alihasan charity al ihsan charitable society elehssan society elehssan bir wa elehssan society birr and elehssan society al birr wa al ihsan wa al naqa al ahsan 

[Person] Anatoliy Anatolievich Bulavko  (score=1165, datasets=12)
  anatoli

Part B template: 15 rows
query_id  query_type                                                                                                                                                                   query_text          expected_doc_ids                                                                                                                                      notes
  Q3_001           3                                                                                                                                                                              NK-355Sbv9ZUXJLakBMDPJHk7                       keywords: crude, launder, oil, prohibit, sanction | caption hint: "Mr Murtaza Ali Lakhani" [Person] (ca, mc, pk, gb)
  Q3_002           3                                                                                                                                                                              NK-2prrTYm97q4eUKFwVV3TaH                                     k

Exported Part B template → /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_tem_2/IR_project/data/evaluation/queries_part_b_template.xlsx

Expert instructions:
  Type 3: Fill yellow query_text cells — write a natural language description, NO entity names
  Type 4: Fill yellow query_text cells — frame as a relational question about the hinted entity
  Type 7: query_text is pre-filled; fill gold_answer (yellow) with a 1-2 paragraph narrative

Both files ready:
  Part A (auto): /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_tem_2/IR_project/data/evaluation/queries_part_a.xlsx
  Part B (human): /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_tem_2/IR_project/data/evaluation/queries_part_b_template.xlsx
